# Своя модель: меняем манеру ответа

Основной путь: NVIDIA, Python 3.11, Qwen2.5-1.5B-Instruct и 512 готовых диалогов. Проходим ячейки по порядку в чистом ядре. Установка выполняется один раз, после неё перезапускаем ядро. Контрольный вопрос не содержит инструкции о стиле.

## Установим библиотеки

%pip устанавливает пакеты в Python текущего ядра. Первая команда берёт сборку PyTorch с CUDA 12.1 из официального репозитория. Вторая устанавливает согласованный набор библиотек курса. После установки перезапустите ядро через Kernel → Restart Kernel. Установка выполняется один раз для окружения.

In [1]:
!uv pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install transformers==4.57.1 datasets==4.8.3 peft==0.17.1 trl==0.22.2 accelerate==1.13.0 huggingface-hub==0.36.2 safetensors==0.7.0

Using Python 3.11.15 environment at: /mnt/projects/llm_fine-tuning/.venv
Checked 1 package in 3ms
Using Python 3.11.15 environment at: /mnt/projects/llm_fine-tuning/.venv
Checked 7 packages in 11ms


## Проверим видеокарту

torch.__version__ показывает сборку. cuda.is_available() проверяет, может ли этот Python использовать NVIDIA. assert останавливает основной путь с понятным сообщением, если CUDA недоступна. Номер 0 обозначает первую доступную видеокарту.

In [2]:
# PyTorch выполняет вычисления модели и проверяет доступ к CUDA.
import torch

print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "CUDA недоступна: см. раздел помощи"
print("Видеокарта:", torch.cuda.get_device_name(0))

PyTorch: 2.5.1+cu121
Видеокарта: NVIDIA RTX 4500 Ada Generation


## Выберем модель

Слева от / — владелец репозитория, справа — его имя. Строка совпадает с концом адреса huggingface.co/Qwen/Qwen2.5-1.5B-Instruct. Для другой модели копируют её собственный идентификатор. Здесь пока выбирается только модель.

In [3]:
# Идентификатор копируется из заголовка страницы на Hugging Face.
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

## Загрузим готовый генератор текста

text-generation — задача продолжения текста. model указывает репозиторий. device=0 размещает модель на первой NVIDIA. dtype="auto" использует формат весов из конфигурации; у выбранной Qwen это bfloat16, формат с двумя байтами на число. Внутренний алгоритм внимания и служебные токены выбирает библиотека.

In [4]:
# pipeline объединяет модель, токенизатор и генерацию текста.
from transformers import pipeline

chat = pipeline(
    "text-generation", model=model_id, device=0, dtype="auto"
)

/mnt/projects/llm_fine-tuning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


## Посмотрим на вход модели

Токенизатор переводит строку в идентификаторы фрагментов текста. input_ids — список этих чисел, decode выполняет обратное преобразование. Числа обозначают элементы словаря, а не оценки смысла слов.

In [5]:
# Токенизатор уже загружен вместе с моделью.
ids = chat.tokenizer("Hello, captain!")["input_ids"]
print(ids)
print(chat.tokenizer.decode(ids))

[9707, 11, 23915, 0]
Hello, captain!


## Зададим вопрос

role="user" обозначает пользователя, content содержит вопрос. Pipeline сам оформляет диалог для Qwen. max_new_tokens ограничивает продолжение 100 токенами, do_sample=False отключает случайную выборку. Из результата берём первый диалог и последнее сообщение — ответ помощника. Теперь ask("вопрос") вызывается одной строкой. before сохраняет исходный ответ.

In [6]:
# Эта функция избавляет от повторения формата диалога.
def ask(question):
    reply = chat(
        [{"role": "user", "content": question}],
        max_new_tokens=100, do_sample=False
    )
    return reply[0]["generated_text"][-1]["content"]

question = "How can I organize my study time?"
before = ask(question)
print(before)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Organizing your study time effectively is crucial for achieving academic success and maintaining productivity. Here are some tips to help you manage your study sessions more efficiently:

1. **Set Clear Goals**: Define what you want to achieve with each study session. Break down larger goals into smaller, manageable tasks.

2. **Create a Schedule**: Use a planner or digital calendar to schedule your study times. Allocate specific blocks of time dedicated solely to studying. Stick to this schedule as much as possible.

3. **Prior


## Возьмём готовые диалоги

load_dataset открывает набор как таблицу. train — имя раздела, а не команда начала обучения. В каждой строке messages — реплики. Фильтр оставляет короткие вопросы, select берёт первые 512 подходящих строк. Ответы не переписываются, стиль не добавляется вручную.

In [7]:
from datasets import load_dataset

# В наборе уже есть вопросы и ответы в пиратском стиле.
data = load_dataset("winglian/pirate-ultrachat-10k", split="train")

# Возьмём 512 примеров с вопросами короче 300 символов.
data = data.filter(lambda row: len(row["messages"][0]["content"]) < 300)
data = data.select(range(512))
print(data[0])

{'messages': [{'content': 'Create a thank you card to an organization that helped your community in a time of crisis.', 'role': 'user'}, {'content': "Ahoy, Mateys! \n\nWe, the scurvy dogs of [Your Community Name], be hoistin' the Jolly Roger of gratitude for yer generous assistance in our time of need. \n\nYer kindness be like a treasure chest full of doubloons to us, savin' our ship from the stormy seas of hardship. We be forever indebted to yer valiant efforts, aye! \n\nMay yer sails be full, yer grog be strong, and yer treasure chests overflow with gold! We be eternally grateful, mateys!\n\nYours in plunder and good cheer,\n\n[Your Name]\n", 'role': 'assistant'}]}


## Добавим обучаемую часть

PEFT — библиотека экономного дообучения. LoRA добавляет небольшие обучаемые матрицы к преобразованиям модели. CAUSAL_LM обозначает предсказание следующего токена по предыдущим. Стандартная конфигурация PEFT для Qwen выбирает поддерживаемые слои; базовые веса остаются замороженными.

In [8]:
from peft import LoraConfig

# Для первого опыта используем стандартные настройки LoRA.
adapter = LoraConfig(task_type="CAUSAL_LM")

## Зададим размер эксперимента

SFT — обучение на примерах желаемого поведения. Здесь: 400 обновлений; learning_rate=0.0003 задаёт масштаб изменения; по 2 диалога обрабатываются за проход, 4 прохода накапливаются перед обновлением. Это 8 примеров на шаг. max_length ограничивает диалог 384 токенами, длинный хвост обрезается. output_dir — папка результатов, report_to="none" отключает внешние сервисы журналирования. Остальные параметры берутся из выбранной версии TRL.

In [9]:
from trl import SFTConfig, SFTTrainer

settings = SFTConfig(
    output_dir="../outputs/pirate-output",
    max_steps=400, learning_rate=3e-4,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    max_length=384,
    report_to="none"
)

## Запустим дообучение

Передаём загруженную модель, её токенизатор, диалоги, LoRA и настройки. SFTTrainer готовит последовательности, собирает порции данных, вычисляет ошибку и обновляет параметры адаптера. train() запускает цикл. Прогресс появляется в выводе ячейки; loss — ошибка предсказания токенов, а не доля правильных ответов.

In [10]:
trainer = SFTTrainer(
    model=chat.model, processing_class=chat.tokenizer,
    train_dataset=data, peft_config=adapter, args=settings
)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.196300
20,1.970500
30,1.871100
40,1.812900
50,1.736900
60,1.594600
70,1.613400
80,1.613900
90,1.598200
100,1.519500


TrainOutput(global_step=400, training_loss=1.5390160989761352, metrics={'train_runtime': 307.6211, 'train_samples_per_second': 10.402, 'train_steps_per_second': 1.3, 'total_flos': 9205263906852864.0, 'train_loss': 1.5390160989761352, 'epoch': 6.25})

## Посмотрим на ход обучения

В log_history есть измерения loss и итоговая статистика. Берём записи с loss и печатаем шаг и усреднённую ошибку. График в лекции строится по такому журналу реального запуска; учебная анимация показана отдельно.

In [11]:
# Журнал доступен без подключения внешнего сервиса.
for row in trainer.state.log_history:
    if "loss" in row:
        print(row["step"], round(row["loss"], 3))

10 2.196
20 1.97
30 1.871
40 1.813
50 1.737
60 1.595
70 1.613
80 1.614
90 1.598
100 1.52
110 1.532
120 1.553
130 1.538
140 1.521
150 1.476
160 1.51
170 1.457
180 1.512
190 1.53
200 1.494
210 1.461
220 1.403
230 1.443
240 1.513
250 1.51
260 1.444
270 1.464
280 1.448
290 1.404
300 1.478
310 1.442
320 1.445
330 1.435
340 1.447
350 1.448
360 1.457
370 1.402
380 1.445
390 1.399
400 1.426


## Повторим тот же вопрос

trainer.model содержит базу с адаптером. eval() включает режим использования. Вопрос и настройки генерации не меняются, инструкции «говори как пират» нет. Сравниваем сохранённый before с after.

In [12]:
# Подключаем обученный адаптер к тому же генератору.
chat.model = trainer.model
chat.model.eval()

after = ask(question)
print("ДО:\n", before)
print("ПОСЛЕ:\n", after)

ДО:
 Organizing your study time effectively is crucial for achieving academic success and maintaining productivity. Here are some tips to help you manage your study sessions more efficiently:

1. **Set Clear Goals**: Define what you want to achieve with each study session. Break down larger goals into smaller, manageable tasks.

2. **Create a Schedule**: Use a planner or digital calendar to schedule your study times. Allocate specific blocks of time dedicated solely to studying. Stick to this schedule as much as possible.

3. **Prior
ПОСЛЕ:
 Ahoy there, matey!  Organizing yer study time be like rigging a ship for a stormy sea voyage - it requires planning and strategy!

First, ye need to know what's on deck, eh? What subjects be callin' fer attention? A map of the curriculum be handy, but don't forget to include some extra-curricular activities too, like practice drills or treasure hunts.

Next, decide where ye'll be sailin'. Find a quiet spot with no distractions,


## Проверим другой вопрос

Первый вопрос проверяет перенос стиля на другую тему, второй — на русский язык. Английский набор не гарантирует такой перенос. Смотреть нужно и на стиль, и на смысл: пиратское приветствие само по себе не делает ответ хорошим.

In [13]:
print(ask("Explain what a neural network is in two sentences."))
print(ask("Скажи, как лучше учиться программированию?"))

Ahoy there, matey! A neural network be like a treasure map for the brain's circuits, helping it navigate through complex puzzles and make decisions as smart as a parrot with a deck of cards! 

Ahoy there, matey!  Learning programming be like finding buried treasure on the high seas - it's all about patience and perseverance! Here be some tips to make your journey smoother:

1. **Start with basics**: Learn the fundamentals of coding, like variables, loops, and functions. It'll help you build a sturdy ship for navigating through the code.
2. **Practice, practice, practice**: The more you write code, the better you get. Try building simple programs first, then


## Сохраним результат

В pirate-adapter записываются веса LoRA, конфигурация и токенизатор. Базовая модель не копируется: при следующем запуске нужны и база, и адаптер. База уже находится в локальном кэше. Отдельный блокнот показывает загрузку в новом сеансе.

In [14]:
trainer.save_model("../adapters/pirate-adapter")
chat.tokenizer.save_pretrained("../adapters/pirate-adapter")

('../adapters/pirate-adapter/tokenizer_config.json',
 '../adapters/pirate-adapter/special_tokens_map.json',
 '../adapters/pirate-adapter/chat_template.jinja',
 '../adapters/pirate-adapter/vocab.json',
 '../adapters/pirate-adapter/merges.txt',
 '../adapters/pirate-adapter/added_tokens.json',
 '../adapters/pirate-adapter/tokenizer.json')